# 03 - Baseline Models

This notebook trains three simple baseline models for the binary heart disease classification task:

- Logistic Regression
- Random Forest
- XGBoost

The goal is to get a clean first modeling run and compare basic metrics. No hyperparameter tuning is performed here.

## 1. Imports and Paths

The models use the processed train/test files created by the preprocessing notebook.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluate import (
    evaluate_model,
    plot_confusion_matrix,
    plot_roc_curve,
    write_results_markdown,
)

In [ ]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR

## 2. Load Processed Data

The feature files already contain imputed numerical columns and one-hot encoded categorical columns. The label files contain the binary `target` column.

In [ ]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze("columns")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert y_train.isna().sum() == 0
assert y_test.isna().sum() == 0

## 3. Define Baseline Models

These models use straightforward default-style settings. Random seeds are fixed for reproducibility, and Logistic Regression receives a higher `max_iter` only to avoid premature convergence issues.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    ),
}

models

## 4. Train and Evaluate

Each baseline is fit on the training data and evaluated on the held-out test set using accuracy, precision, recall, F1, ROC-AUC, and confusion-matrix counts.

For medical risk prediction, recall and false negatives are especially important because false negatives are patients with heart disease who are incorrectly classified as low-risk.

In [ ]:
metric_columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "False Negatives",
]

In [ ]:
results = []

for model_name, model in models.items():
    results.append(evaluate_model(model_name, model, X_train, y_train, X_test, y_test))

results_df = pd.DataFrame(results)
results_df[metric_columns]

## 5. Baseline Comparison and Outputs

The table below sorts the baseline results by ROC-AUC. The notebook also saves a Markdown report and two figures for later documentation.

In [ ]:
results_df.sort_values("ROC-AUC", ascending=False).reset_index(drop=True)[metric_columns]

In [ ]:
write_results_markdown(results_df, REPORTS_DIR / "results.md")

confusion_fig, _ = plot_confusion_matrix(
    models,
    X_test,
    y_test,
    save_path=FIGURES_DIR / "baseline_confusion_matrices.png",
)

roc_fig, _ = plot_roc_curve(
    models,
    X_test,
    y_test,
    save_path=FIGURES_DIR / "baseline_roc_curve.png",
)

Generated outputs:

- `reports/results.md`
- `reports/figures/baseline_confusion_matrices.png`
- `reports/figures/baseline_roc_curve.png`